In [6]:
!apt-get install -y nodejs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  javascript-common libc-ares2 libjs-highlight.js libnode72 nodejs-doc
Suggested packages:
  apache2 | lighttpd | httpd npm
The following NEW packages will be installed:
  javascript-common libc-ares2 libjs-highlight.js libnode72 nodejs nodejs-doc
0 upgraded, 6 newly installed, 0 to remove and 53 not upgraded.
Need to get 13.7 MB of archives.
After this operation, 54.0 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 javascript-common all 11+nmu1 [5,936 B]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libjs-highlight.js all 9.18.5+dfsg1-1 [367 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libc-ares2 amd64 1.18.1-1ubuntu0.22.04.3 [45.1 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libnode72 amd64 12.22.9~dfsg-1ubuntu3.6 [10.8 MB]


In [15]:
!pip install ultralytics yt-dlp huggingface_hub pytubefix

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.0 MB/s eta 0:00:00


In [3]:
import os
import cv2
import json
import math
import yt_dlp
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download, login
from scipy.signal import find_peaks, savgol_filter

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [19]:
from pytubefix import YouTube
from pytubefix.cli import on_progress

url = "https://www.youtube.com/watch?v=YZX97v8FaVg"

yt = YouTube(url, use_oauth=True, allow_oauth_cache=False, on_progress_callback=on_progress)
ys = yt.streams.get_highest_resolution()
ys.download()

Please open https://www.google.com/device and input code SFX-LML-RPH
Press enter when you have completed this step.
Please open https://www.google.com/device and input code JJN-XLG-YXR
Press enter when you have completed this step.


HTTPError: HTTP Error 428: Precondition Required

In [12]:
# Cell 2 — Download
import yt_dlp, os

youtube_url = input('Paste YouTube URL: ')

ydl_opts = {
    'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/bestvideo+bestaudio/best',
    'merge_output_format': 'mp4',
    'outtmpl': '%(title)s.%(ext)s',
    'quiet': False,
    'cookiefile': 'fyp-youtube.txt',
    # Drop the player_client override — let yt-dlp pick defaults now that node works
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(youtube_url, download=True)
    VIDEO_PATH = ydl.prepare_filename(info)
    if not os.path.exists(VIDEO_PATH):
        VIDEO_PATH = VIDEO_PATH.rsplit('.', 1)[0] + '.mp4'

print(f'Downloaded: {VIDEO_PATH}')
print(f'File size:  {os.path.getsize(VIDEO_PATH) / 1e6:.1f} MB')

Paste YouTube URL: https://www.youtube.com/watch?v=YZX97v8FaVg&t=346s
[youtube] Extracting URL: https://www.youtube.com/watch?v=YZX97v8FaVg&t=346s
[youtube] YZX97v8FaVg: Downloading webpage
[youtube] YZX97v8FaVg: Downloading tv downgraded player API JSON


ERROR: [youtube] YZX97v8FaVg: Requested format is not available. Use --list-formats for a list of available formats


DownloadError: ERROR: [youtube] YZX97v8FaVg: Requested format is not available. Use --list-formats for a list of available formats

In [ ]:
login(token="")
path = hf_hub_download(
    repo_id="chathushik/badminton-pose-estimation",
    filename="best.pt"
)
custom_model = YOLO(path)
baseline_model = YOLO('yolov8n-pose.pt')

In [ ]:
# MODEL_PATH = "/kaggle/input/models/hirushafernando/badminton-pose-detection/pytorch/default/1/best.pt"
# custom_model = YOLO(MODEL_PATH)
# baseline_model = YOLO('yolov8n-pose.pt')

## Player Tracking With Both Fine-Tuned & Baseline Model Combination

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# MULTIPLE PLAYER TRACKING WITH BOTH FINE-TUNED & BASELINE MODEL COMBINATION
# (Single-player target locking applied — correct keypoints via baseline model)
# ──────────────────────────────────────────────────────────────────────────────

# --- CONFIG ---
OUTPUT_JSON  = "analyzed_single_player_badminton.json"

CONF_THRES          = 0.35
KEYPOINT_CONF_THRES = 0.25

MAX_MISSING_FRAMES  = 40
MAX_CENTER_DISTANCE = 200
MIN_MATCH_SCORE     = 0.25


COURT_Y_MIN = 0.28   # feet must be below 28% of frame height
COURT_Y_MAX = 0.62   # feet must be above 62% of frame height  (excludes near player)

# Motion detection: a candidate must have at least this fraction of its crop
# covered by moving pixels (frame-diff) to be considered an active player.
MOTION_THRESHOLD = 0.04   # 4% of crop pixels must be moving

KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle"
]

SKELETON = [
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (5, 6),
    (5, 11), (6, 12),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (0, 1), (0, 2),
    (1, 3), (2, 4)
]


In [ ]:

# ── Helper functions ───────────────────────────────────────────────────────────

def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]);  yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]);  yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
    areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])
    return inter / (areaA + areaB - inter + 1e-6)

def bbox_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2])

def center_distance(boxA, boxB):
    return np.linalg.norm(bbox_center(boxA) - bbox_center(boxB))

def bbox_area(box):
    x1, y1, x2, y2 = box
    return max(0, x2 - x1) * max(0, y2 - y1)

def safe_crop(frame, box):
    h, w = frame.shape[:2]
    x1 = int(max(0, min(w - 1, box[0])))
    y1 = int(max(0, min(h - 1, box[1])))
    x2 = int(max(0, min(w - 1, box[2])))
    y2 = int(max(0, min(h - 1, box[3])))
    if x2 <= x1 or y2 <= y1:
        return None
    return frame[y1:y2, x1:x2]

def color_histogram(frame, box):
    crop = safe_crop(frame, box)
    if crop is None or crop.size == 0:
        return None
    crop = cv2.resize(crop, (64, 128))
    hsv  = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1], None, [32, 32], [0, 180, 0, 256])
    cv2.normalize(hist, hist)
    return hist

def histogram_similarity(histA, histB):
    if histA is None or histB is None:
        return 0.0
    return max(0.0, min(1.0, cv2.compareHist(histA, histB, cv2.HISTCMP_CORREL)))

def motion_score(frame, prev_frame, box):
    """Fraction of bbox crop pixels that changed significantly between frames."""
    if prev_frame is None:
        return 1.0   # no prior frame → don't penalise on first lock
    crop_curr = safe_crop(frame,      box)
    crop_prev = safe_crop(prev_frame, box)
    if crop_curr is None or crop_prev is None:
        return 0.0
    if crop_curr.shape != crop_prev.shape:
        crop_prev = cv2.resize(crop_prev, (crop_curr.shape[1], crop_curr.shape[0]))
    diff  = cv2.absdiff(crop_curr, crop_prev)
    gray  = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    moved = np.count_nonzero(gray > 25)          # pixels that changed by >25 intensity
    return moved / (gray.size + 1e-6)

def is_inside_court(box, frame_h):
    """
    True only if the detection's feet (bbox bottom) fall within the far-player
    court band. Rejects spectators above the back wall AND the near-side player.
    """
    foot_y = box[3] / frame_h          # normalised y of bbox bottom edge
    return COURT_Y_MIN < foot_y < COURT_Y_MAX

def face_visibility_score(kpt_conf):
    score  = sum(2 for i in [0, 1, 2, 3, 4] if kpt_conf[i] > KEYPOINT_CONF_THRES)
    score += sum(1 for i in [5, 6]           if kpt_conf[i] > KEYPOINT_CONF_THRES)
    return score

def extract_detections(result, frame):
    detections = []
    if result.boxes is None or result.keypoints is None:
        return detections
    boxes     = result.boxes.xyxy.cpu().numpy()
    det_confs = result.boxes.conf.cpu().numpy()
    kpts_xy   = result.keypoints.xy.cpu().numpy()
    kpts_conf = result.keypoints.conf.cpu().numpy() \
                if result.keypoints.conf is not None \
                else np.ones((len(boxes), 17))
    for i in range(len(boxes)):
        if det_confs[i] < CONF_THRES:
            continue
        detections.append({
            "box":           boxes[i].tolist(),
            "det_conf":      float(det_confs[i]),
            "keypoints":     kpts_xy[i].tolist(),
            "keypoint_conf": kpts_conf[i].tolist(),
            "hist":          color_histogram(frame, boxes[i]),
        })
    return detections

def select_initial_target(detections, frame, prev_frame, frame_h):
    """
    Pick the best candidate using three hard filters then a score:
      HARD 1 — feet must be inside the far-player court band (COURT_Y_MIN..COURT_Y_MAX)
      HARD 2 — must show meaningful motion vs previous frame (not a static spectator)
    SCORE   — face visibility (primary) + detection confidence + bbox size
    """
    best_det, best_score = None, -1

    for det in detections:
        box      = det["box"]
        kpt_conf = np.array(det["keypoint_conf"])

        # Hard filter 1: court boundary
        if not is_inside_court(box, frame_h):
            continue

        # Hard filter 2: must be moving
        mv = motion_score(frame, prev_frame, box)
        if mv < MOTION_THRESHOLD:
            continue

        score = (
            face_visibility_score(kpt_conf)                  * 4.0 +
            det["det_conf"]                                  * 2.0 +
            min(bbox_area(box) / (frame_h * frame_h), 1.0)  * 1.0 +
            min(mv / 0.20, 1.0)                              * 1.0   # more motion = better
        )
        if score > best_score:
            best_score, best_det = score, det

    return best_det

def match_locked_target(detections, frame, prev_frame, locked_box, locked_hist, frame_h):
    """
    Re-identify the locked player each frame.
    Hard filter: still must be inside the court band.
    Scoring: IoU + distance + appearance + motion.
    """
    best_det, best_score = None, -1

    for det in detections:
        box = det["box"]

        if not is_inside_court(box, frame_h):
            continue

        iou  = compute_iou(locked_box, box)
        dist = center_distance(locked_box, box)
        if dist > MAX_CENTER_DISTANCE and iou < 0.05:
            continue

        area_ratio  = bbox_area(box) / (bbox_area(locked_box) + 1e-6)
        mv          = motion_score(frame, prev_frame, box)

        match_score = (
            iou                                              * 4.0 +
            max(0.0, 1.0 - dist / MAX_CENTER_DISTANCE)      * 2.0 +
            histogram_similarity(locked_hist, det["hist"])   * 3.0 +
            det["det_conf"]                                  * 1.0 +
            (1.0 - min(abs(1.0 - area_ratio), 1.0))         * 1.0
        ) / 11.0

        if match_score > best_score:
            best_score, best_det = match_score, det

    return (None, best_score) if best_score < MIN_MATCH_SCORE else (best_det, best_score)

def draw_pose(frame, kpts, kpt_conf):
    for p1, p2 in SKELETON:
        if kpt_conf[p1] > KEYPOINT_CONF_THRES and kpt_conf[p2] > KEYPOINT_CONF_THRES:
            cv2.line(frame,
                     (int(kpts[p1][0]), int(kpts[p1][1])),
                     (int(kpts[p2][0]), int(kpts[p2][1])),
                     (0, 255, 255), 2)
    for i, (x, y) in enumerate(kpts):
        if kpt_conf[i] > KEYPOINT_CONF_THRES:
            cv2.circle(frame, (int(x), int(y)), 4, (0, 0, 255), -1)

## Use Inference Function

In [ ]:

# ── Main processing loop ───────────────────────────────────────────────────────

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), "Cannot open video"

frame_w      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps          = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))


tracking_json = {
    "video_name":      VIDEO_PATH,
    "baseline_model":  "yolov8n-pose.pt",
    "custom_model":    "best.pt (badminton fine-tuned)",
    "tracking_method": "baseline keypoints + court boundary + motion filter + appearance lock",
    "court_y_min":     COURT_Y_MIN,
    "court_y_max":     COURT_Y_MAX,
    "fps":             fps,
    "frame_width":     frame_w,
    "frame_height":    frame_h,
    "frames":          []
}

# Tracking state
locked              = False
locked_box          = None
locked_hist         = None
missing_count       = 0
last_good_detection = None
pose_name           = "Detecting..."
prev_frame          = None          # for motion detection


In [ ]:

print(f"Processing {total_frames} frames...")

for frame_id in tqdm(range(total_frames)):
    ret, frame = cap.read()
    if not ret:
        break

    timestamp = frame_id / fps

    # ── Step 1: baseline model → correct keypoints for ALL detected people ──
    base_result = baseline_model(frame, conf=CONF_THRES, verbose=False)[0]
    detections  = extract_detections(base_result, frame)

    # ── Step 2: custom model → shot classification (frame-level) ───────────
    custom_result = custom_model(frame, verbose=False)[0]
    if custom_result.boxes and len(custom_result.boxes) > 0:
        pose_id   = int(custom_result.boxes.cls[0].item())
        pose_name = custom_result.names[pose_id]
        shot_conf = float(custom_result.boxes.conf[0].item())
    else:
        shot_conf = 0.0

    # ── Step 3: lock onto / track the single target player ─────────────────
    selected    = None
    match_score = None

    if not locked:
        if detections:
            selected = select_initial_target(detections, frame, prev_frame, frame_h)
            if selected is not None:
                locked              = True
                locked_box          = selected["box"]
                locked_hist         = selected["hist"]
                last_good_detection = selected
                missing_count       = 0
    else:
        selected, match_score = match_locked_target(
            detections, frame, prev_frame, locked_box, locked_hist, frame_h
        )
        if selected is not None:
            locked_box = selected["box"]
            new_hist   = selected["hist"]
            if locked_hist is not None and new_hist is not None:
                locked_hist = cv2.addWeighted(locked_hist, 0.85, new_hist, 0.15, 0)
            last_good_detection = selected
            missing_count       = 0
        else:
            missing_count += 1
            if missing_count <= MAX_MISSING_FRAMES:
                selected = last_good_detection
            else:
                locked = False
                locked_box = locked_hist = last_good_detection = None
                missing_count = 0

    # ── Step 4: annotate frame ──────────────────────────────────────────────
    frame_data = {
        "frame_id":            frame_id,
        "timestamp":           round(timestamp, 4),
        "player_detected":     False,
        "tracking_status":     "missing",
        "match_score":         match_score,
        "shot_classification": pose_name,
        "shot_confidence":     shot_conf,
        "bounding_box":        None,
        "detection_confidence": None,
        "keypoints":           []
    }

    if selected is not None:
        box     = selected["box"]
        kpts    = np.array(selected["keypoints"])
        kpt_conf = np.array(selected["keypoint_conf"])

        x1, y1, x2, y2 = map(int, box)
        status    = "tracked" if missing_count == 0 else "temporarily_predicted"

        # JSON
        frame_data["player_detected"]      = True
        frame_data["tracking_status"]      = status
        frame_data["detection_confidence"] = float(selected["det_conf"])
        frame_data["bounding_box"] = {
            "x1": float(box[0]), "y1": float(box[1]),
            "x2": float(box[2]), "y2": float(box[3]),
            "width":  float(box[2] - box[0]),
            "height": float(box[3] - box[1]),
        }
        for idx, name in enumerate(KEYPOINT_NAMES):
            frame_data["keypoints"].append({
                "id":         idx,
                "name":       name,
                "x":          float(kpts[idx][0]),
                "y":          float(kpts[idx][1]),
                "confidence": float(kpt_conf[idx]),
            })

    tracking_json["frames"].append(frame_data)
    prev_frame = frame.copy()

cap.release()

In [ ]:
with open(OUTPUT_JSON, "w") as f:
    json.dump(tracking_json, f, indent=4)

print("✅ Done")
print("JSON: ", OUTPUT_JSON)

In [ ]:
JSON_PATH = OUTPUT_JSON

with open(JSON_PATH, 'r') as f:
    data = json.load(f)

FPS         = data.get('fps', 60)
FRAME_W     = data.get('frame_width', 1920)
FRAME_H     = data.get('frame_height', 1080)
frames_raw  = data.get('frames', [])

# ── Calibration: pixels → metres ──────────────────────────────────────────────
# Standard badminton court is 13.4 m long × 6.1 m wide.
# We assume the visible court spans the full frame height (long axis).
# Adjust COURT_M_HEIGHT / COURT_M_WIDTH to match your camera crop.
COURT_M_HEIGHT = 13.4   # metres visible in frame height direction
COURT_M_WIDTH  = 6.1    # metres visible in frame width direction
PX_PER_M_Y     = FRAME_H / COURT_M_HEIGHT
PX_PER_M_X     = FRAME_W / COURT_M_WIDTH

print(f'FPS={FPS}, frames={len(frames_raw)}, resolution={FRAME_W}x{FRAME_H}')
print(f'Calibration: {PX_PER_M_X:.1f} px/m (X)  {PX_PER_M_Y:.1f} px/m (Y)')


In [ ]:
KEYPOINT_NAMES = [
    'nose','left_eye','right_eye','left_ear','right_ear',
    'left_shoulder','right_shoulder','left_elbow','right_elbow',
    'left_wrist','right_wrist','left_hip','right_hip',
    'left_knee','right_knee','left_ankle','right_ankle'
]
KP_IDX = {name: i for i, name in enumerate(KEYPOINT_NAMES)}
CONF_THR = 0.25

rows = []
for fr in frames_raw:
    if not fr.get('player_detected', False):
        continue

    bb  = fr['bounding_box']
    kps = {kp['name']: kp for kp in fr.get('keypoints', [])}

    def kp_xy(name):
        """Return (x, y) or (nan, nan) if low confidence."""
        k = kps.get(name)
        if k and k['confidence'] >= CONF_THR:
            return k['x'], k['y']
        return np.nan, np.nan

    # Ankle midpoint = foot position on court
    lax, lay = kp_xy('left_ankle')
    rax, ray = kp_xy('right_ankle')
    foot_x   = np.nanmean([lax, rax])
    foot_y   = np.nanmean([lay, ray])

    # Hip midpoint = centre-of-mass proxy
    lhx, lhy = kp_xy('left_hip')
    rhx, rhy = kp_xy('right_hip')
    hip_x    = np.nanmean([lhx, rhx])
    hip_y    = np.nanmean([lhy, rhy])

    # Wrist positions (for lateral reach)
    lwx, lwy = kp_xy('left_wrist')
    rwx, rwy = kp_xy('right_wrist')

    rows.append(dict(
        frame_id   = fr['frame_id'],
        timestamp  = fr['timestamp'],
        bb_x1=bb['x1'], bb_y1=bb['y1'], bb_x2=bb['x2'], bb_y2=bb['y2'],
        bb_w=bb['width'], bb_h=bb['height'],
        foot_x=foot_x, foot_y=foot_y,
        hip_x=hip_x,  hip_y=hip_y,
        lax=lax, lay=lay, rax=rax, ray=ray,   # individual ankles
        lwx=lwx, lwy=lwy, rwx=rwx, rwy=rwy,  # wrists
        lhx=lhx, lhy=lhy, rhx=rhx, rhy=rhy,  # individual hips
    ))

df = pd.DataFrame(rows).sort_values('frame_id').reset_index(drop=True)
print(f'Clean frames with detections: {len(df)}')
df.head(3)


In [ ]:
# Speed = displacement of foot midpoint between consecutive frames ÷ time
# Smooth positions first to reduce jitter from keypoint noise.

WIN = max(3, int(FPS * 0.10))  # 100 ms smoothing window (must be odd)
WIN = WIN if WIN % 2 == 1 else WIN + 1

df['foot_x_sm'] = savgol_filter(df['foot_x'].ffill().bfill(), WIN, 2)
df['foot_y_sm'] = savgol_filter(df['foot_y'].ffill().bfill(), WIN, 2)

dx_px = df['foot_x_sm'].diff()                     # pixel displacement X
dy_px = df['foot_y_sm'].diff()                     # pixel displacement Y
dt    = df['timestamp'].diff().replace(0, np.nan)  # seconds

dx_m  = dx_px / PX_PER_M_X
dy_m  = dy_px / PX_PER_M_Y

df['speed_mps'] = np.sqrt(dx_m**2 + dy_m**2) / dt
df['speed_mps'] = df['speed_mps'].clip(upper=15)   # cap unrealistic spikes (>15 m/s)

print(f"Speed  →  mean: {df['speed_mps'].mean():.2f} m/s  |  max: {df['speed_mps'].max():.2f} m/s")


In [ ]:
speed_sm = savgol_filter(df['speed_mps'].fillna(0), WIN, 2)
df['accel_mps2'] = pd.Series(speed_sm).diff() / dt
df['accel_mps2'] = df['accel_mps2'].clip(-30, 30)   # ±30 m/s² is physiological ceiling

print(f"Accel  →  mean: {df['accel_mps2'].mean():.2f} m/s²  |  max_burst: {df['accel_mps2'].max():.2f} m/s²")


In [ ]:
# Divide the court into a 3×3 grid (lateral: Left / Centre / Right, depth: Net / Mid / Back)
# foot_x  → lateral zone,  foot_y → depth zone
# Adjust thresholds to match your camera perspective.

def lateral_zone(x):
    if pd.isna(x):    return 'Unknown'
    frac = x / FRAME_W
    if frac < 0.33:   return 'Left'
    elif frac < 0.67: return 'Centre'
    else:             return 'Right'

def depth_zone(y):
    """Near-net = low y (top of frame), back-court = high y."""
    if pd.isna(y):    return 'Unknown'
    frac = y / FRAME_H
    if frac < 0.33:   return 'Net'
    elif frac < 0.67: return 'Mid'
    else:             return 'Back'

df['lateral_zone'] = df['foot_x'].apply(lateral_zone)
df['depth_zone']   = df['foot_y'].apply(depth_zone)
df['court_zone']   = df['lateral_zone'] + '-' + df['depth_zone']

zone_counts = df['court_zone'].value_counts()
print('Court Zone distribution (frames):')
print(zone_counts.to_string())


In [ ]:
# Strategy: detect ankle oscillations in the Y direction.
# Each full up-down-up cycle of the ankle = one stride.
# We count peaks (ankle at highest point = foot lifted) per second.

left_y  = df['lay'].ffill().bfill().values
right_y = df['ray'].ffill().bfill().values

# In image coordinates Y increases downward, so foot lift = local minimum in Y
min_dist_frames = int(FPS * 0.15)   # strides no faster than ~3.3 Hz

left_peaks,  _ = find_peaks(-left_y,  distance=min_dist_frames, prominence=5)
right_peaks, _ = find_peaks(-right_y, distance=min_dist_frames, prominence=5)

total_peaks   = len(left_peaks) + len(right_peaks)
total_seconds = df['timestamp'].max() - df['timestamp'].min()
stride_freq   = total_peaks / total_seconds if total_seconds > 0 else 0

print(f'Left ankle lifts : {len(left_peaks)}')
print(f'Right ankle lifts: {len(right_peaks)}')
print(f'Stride Frequency : {stride_freq:.2f} Hz  ({total_seconds:.1f}s clip)')

# Per-second rolling stride rate
df['stride_freq_hz'] = np.nan
all_peaks = sorted(np.concatenate([left_peaks, right_peaks]))
peak_times = df['timestamp'].iloc[all_peaks].values

for i, row in df.iterrows():
    t = row['timestamp']
    count = np.sum((peak_times >= t - 0.5) & (peak_times <= t + 0.5))
    df.at[i, 'stride_freq_hz'] = count   # peaks per 1-second window


In [ ]:
# Jump height = upward displacement of hip midpoint from its rolling baseline.
# A large upward movement (low Y value in image) means the player has jumped.

hip_y_series = df['hip_y'].ffill().bfill()

# Rolling baseline: 1-second window median as the "standing" reference
baseline_window = max(3, int(FPS))
hip_baseline    = hip_y_series.rolling(baseline_window, center=True, min_periods=1).median()

# Jump = hip moves ABOVE baseline (lower Y than baseline in image coords)
df['jump_height_px'] = (hip_baseline - hip_y_series).clip(lower=0)

# Detect jump events (peak of jump height signal)
jump_peaks, jump_props = find_peaks(
    df['jump_height_px'],
    height=10,           # minimum 10 px lift to count as a jump
    distance=int(FPS * 0.3),
    prominence=8
)

peak_heights = df['jump_height_px'].iloc[jump_peaks].values

print(f'Jumps detected : {len(jump_peaks)}')
if len(peak_heights) > 0:
    print(f'Max jump height: {peak_heights.max():.1f} px')
    print(f'Avg jump height: {peak_heights.mean():.1f} px')
    print(f'(Approx metres at {PX_PER_M_Y:.1f} px/m: max={peak_heights.max()/PX_PER_M_Y:.2f} m)')


In [ ]:
# Recovery Speed = average speed in the 0.5 s window AFTER a peak speed burst.
# A burst is defined as speed > 75th percentile.

BURST_THRESHOLD = df['speed_mps'].quantile(0.75)
POST_BURST_S    = 0.5   # seconds to measure recovery over
POST_FRAMES     = int(POST_BURST_S * FPS)

burst_frames, _ = find_peaks(
    df['speed_mps'].fillna(0),
    height=BURST_THRESHOLD,
    distance=int(FPS * 0.3)
)

recovery_speeds = []
for bf in burst_frames:
    end_idx = min(bf + POST_FRAMES, len(df) - 1)
    window  = df['speed_mps'].iloc[bf:end_idx]
    recovery_speeds.append(window.mean())

df['is_burst'] = False
df.loc[df.index[burst_frames], 'is_burst'] = True

avg_recovery_speed = np.mean(recovery_speeds) if recovery_speeds else np.nan

print(f'Speed bursts (>{BURST_THRESHOLD:.2f} m/s): {len(burst_frames)}')
print(f'Mean recovery speed (0.5s post-burst): {avg_recovery_speed:.2f} m/s')


In [ ]:
# Lateral Reach = maximum horizontal distance between the two wrists.
# Computed per frame; then we report the distribution.

df['lat_reach_px'] = abs(df['rwx'] - df['lwx'])   # pixel span of wrists
df['lat_reach_m']  = df['lat_reach_px'] / PX_PER_M_X

# Also compute single-arm reach: max X distance from hip centre to either wrist
df['hip_cx']          = (df['lhx'].fillna(df['rhx']) + df['rhx'].fillna(df['lhx'])) / 2
df['left_reach_m']    = abs(df['lwx'] - df['hip_cx']) / PX_PER_M_X
df['right_reach_m']   = abs(df['rwx'] - df['hip_cx']) / PX_PER_M_X
df['max_arm_reach_m'] = df[['left_reach_m', 'right_reach_m']].max(axis=1)

print(f"Lateral Reach (wrist span)  →  mean: {df['lat_reach_m'].mean():.2f} m  |  max: {df['lat_reach_m'].max():.2f} m")
print(f"Max single-arm reach        →  mean: {df['max_arm_reach_m'].mean():.2f} m  |  max: {df['max_arm_reach_m'].max():.2f} m")


In [ ]:
# Movement Efficiency Index (MEI)
# MEI = Net displacement (straight-line distance A→B) / Total path length (sum of all steps)
# A score of 1.0 = perfectly straight-line movement. Lower = more wasted movement.
#
# We compute it over a rolling window (e.g. every 2 seconds) to track changes.

WINDOW_S  = 2.0
WINDOW_FR = max(3, int(WINDOW_S * FPS))

foot_x = df['foot_x_sm'].values
foot_y_sm = savgol_filter(df['foot_y'].ffill().bfill(), WIN, 2)

mei_values = []
for i in range(len(df)):
    start = max(0, i - WINDOW_FR)
    xs = foot_x[start:i+1]
    ys = foot_y_sm[start:i+1]

    # Total path length
    dx = np.diff(xs) / PX_PER_M_X
    dy = np.diff(ys) / PX_PER_M_Y
    path_len = np.sum(np.sqrt(dx**2 + dy**2))

    # Net displacement
    net_disp = np.sqrt(((xs[-1] - xs[0]) / PX_PER_M_X)**2 +
                       ((ys[-1] - ys[0]) / PX_PER_M_Y)**2)

    mei = net_disp / path_len if path_len > 0.01 else 1.0
    mei_values.append(min(mei, 1.0))

df['movement_efficiency'] = mei_values

print(f"Movement Efficiency Index  →  mean: {df['movement_efficiency'].mean():.3f}  |  min: {df['movement_efficiency'].min():.3f}")


In [ ]:
summary = pd.DataFrame({
    'Feature': [
        'Player Speed (m/s)',
        'Acceleration (m/s²)',
        'Stride Frequency (Hz)',
        'Jump Height (px)',
        'Recovery Speed (m/s)',
        'Lateral Reach / wrist span (m)',
        'Max Single-Arm Reach (m)',
        'Movement Efficiency Index',
    ],
    'Mean': [
        df['speed_mps'].mean(),
        df['accel_mps2'].mean(),
        df['stride_freq_hz'].mean(),
        df['jump_height_px'].mean(),
        avg_recovery_speed,
        df['lat_reach_m'].mean(),
        df['max_arm_reach_m'].mean(),
        df['movement_efficiency'].mean(),
    ],
    'Max': [
        df['speed_mps'].max(),
        df['accel_mps2'].max(),
        df['stride_freq_hz'].max(),
        df['jump_height_px'].max(),
        np.nan,
        df['lat_reach_m'].max(),
        df['max_arm_reach_m'].max(),
        df['movement_efficiency'].max(),
    ],
    'Min': [
        df['speed_mps'].min(),
        df['accel_mps2'].min(),
        df['stride_freq_hz'].min(),
        df['jump_height_px'].min(),
        np.nan,
        df['lat_reach_m'].min(),
        df['max_arm_reach_m'].min(),
        df['movement_efficiency'].min(),
    ],
    'Std': [
        df['speed_mps'].std(),
        df['accel_mps2'].std(),
        df['stride_freq_hz'].std(),
        df['jump_height_px'].std(),
        np.nan,
        df['lat_reach_m'].std(),
        df['max_arm_reach_m'].std(),
        df['movement_efficiency'].std(),
    ],
})

summary = summary.set_index('Feature').round(3)
print(summary.to_string())


In [ ]:
# Grid: rows = depth (Net / Mid / Back), cols = lateral (Left / Centre / Right)
zone_grid = pd.crosstab(
    df['depth_zone'].replace({'Net':'1-Net','Mid':'2-Mid','Back':'3-Back'}),
    df['lateral_zone']
).reindex(['1-Net','2-Mid','3-Back']).fillna(0).astype(int)   # ← fix: cast to int
zone_grid.index = ['Net','Mid','Back']

plt.figure(figsize=(6, 4))
sns.heatmap(zone_grid, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, cbar_kws={'label': 'Frames'})
plt.title('Court Zone Occupancy Heatmap')
plt.xlabel('Lateral Zone'); plt.ylabel('Depth Zone')
plt.tight_layout()
plt.savefig('court_zone_heatmap.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 14), sharex=True)
t = df['timestamp']

axes[0,0].plot(t, df['speed_mps'], lw=1)
axes[0,0].set_title('Player Speed (m/s)')

axes[0,1].plot(t, df['accel_mps2'], lw=1, color='orange')
axes[0,1].axhline(0, ls='--', lw=0.5, color='grey')
axes[0,1].set_title('Acceleration (m/s²)')

axes[1,0].plot(t, df['stride_freq_hz'], lw=1, color='green')
axes[1,0].set_title('Stride Frequency (Hz)')

axes[1,1].plot(t, df['jump_height_px'], lw=1, color='purple')
axes[1,1].scatter(df['timestamp'].iloc[jump_peaks],
                  df['jump_height_px'].iloc[jump_peaks],
                  color='red', zorder=5, label='Jump peaks')
axes[1,1].legend(fontsize=8)
axes[1,1].set_title('Jump Height (px)')

axes[2,0].plot(t, df['speed_mps'], lw=0.8, alpha=0.5, color='steelblue')
axes[2,0].scatter(df['timestamp'][df['is_burst']],
                  df['speed_mps'][df['is_burst']],
                  color='red', s=20, zorder=5, label='Burst')
axes[2,0].legend(fontsize=8)
axes[2,0].set_title('Recovery Speed — Burst Moments')

axes[2,1].plot(t, df['lat_reach_m'], lw=1, color='teal')
axes[2,1].set_title('Lateral Reach / Wrist Span (m)')

axes[3,0].plot(t, df['max_arm_reach_m'], lw=1, color='brown')
axes[3,0].set_title('Max Single-Arm Reach (m)')

axes[3,1].plot(t, df['movement_efficiency'], lw=1, color='darkgreen')
axes[3,1].set_ylim(0, 1.05)
axes[3,1].set_title('Movement Efficiency Index')

for ax in axes.flat:
    ax.set_xlabel('Time (s)')
    ax.grid(alpha=0.3)

plt.suptitle('Player Movement Feature Extraction — Time Series', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('movement_features_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
output_cols = [
    'frame_id', 'timestamp',
    'speed_mps', 'accel_mps2',
    'court_zone', 'lateral_zone', 'depth_zone',
    'stride_freq_hz',
    'jump_height_px',
    'lat_reach_m', 'max_arm_reach_m',
    'movement_efficiency',
    'is_burst'
]

out_csv = 'movement_features.csv'
df[output_cols].to_csv(out_csv, index=False)
print(f'✅ Saved: {out_csv}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — edit these to match your session
# ─────────────────────────────────────────────────────────────────────────────
MATCH_ID  = 'match_001'
PLAYER_ID = 'player_01'
OUTPUT_JSON_PATH  = 'player_movement_output.json'

# ─────────────────────────────────────────────────────────────────────────────
# HELPER — movement direction angle (degrees, 0=right, 90=up, 180=left)
# ─────────────────────────────────────────────────────────────────────────────
def movement_direction(dx_px, dy_px):
    """Angle in degrees of movement vector. 0 = right, CCW positive."""
    if abs(dx_px) < 1e-3 and abs(dy_px) < 1e-3:
        return 0.0
    # Image Y is inverted so negate dy for standard math angle
    angle = math.degrees(math.atan2(-dy_px, dx_px))
    return round(angle % 360, 1)

# ─────────────────────────────────────────────────────────────────────────────
# HELPER — step type classifier from jump height + speed
# ─────────────────────────────────────────────────────────────────────────────
def classify_step(jump_px, speed, accel):
    if jump_px > 15:
        return 'jump'
    if speed > 4.0:
        return 'sprint'
    if abs(accel) > 5.0:
        return 'lunge'
    if speed > 1.0:
        return 'step'
    return 'stand'

# ─────────────────────────────────────────────────────────────────────────────
# BUILD PER-FRAME ENTRIES
# ─────────────────────────────────────────────────────────────────────────────
# We need the smoothed foot X for direction calculation
dx_sm = df['foot_x_sm'].diff().fillna(0).values
dy_sm_vals = pd.Series(savgol_filter(df['foot_y'].ffill().bfill(), WIN, 2)).diff().fillna(0).values

frame_entries = []

for i, row in df.iterrows():
    # ── Bounding box (top-left x/y + w/h format) ──────────────────────────
    bb = {
        'x':      round(float(row['bb_x1']), 1),
        'y':      round(float(row['bb_y1']), 1),
        'width':  round(float(row['bb_w']),  1),
        'height': round(float(row['bb_h']),  1),
    }

    # ── Keypoints — pull from original tracking JSON for this frame ───────
    src_frame = frames_raw[int(row['frame_id'])] if int(row['frame_id']) < len(frames_raw) else {}
    raw_kps   = src_frame.get('keypoints', [])
    keypoints = [
        {
            'name':       kp['name'],
            'x':          round(float(kp['x']), 1),
            'y':          round(float(kp['y']), 1),
            'confidence': round(float(kp['confidence']), 3),
        }
        for kp in raw_kps
        if kp.get('confidence', 0) >= CONF_THR
    ]

    # ── Court position in metres ──────────────────────────────────────────
    court_x = round(float(row['foot_x']) / PX_PER_M_X, 2) if not math.isnan(row['foot_x']) else None
    court_y = round(float(row['foot_y']) / PX_PER_M_Y, 2) if not math.isnan(row['foot_y']) else None

    # ── Movement features ─────────────────────────────────────────────────
    spd   = round(float(row['speed_mps']),  3) if not math.isnan(row.get('speed_mps', float('nan'))) else 0.0
    acc   = round(float(row['accel_mps2']), 3) if not math.isnan(row.get('accel_mps2', float('nan'))) else 0.0
    dirn  = movement_direction(dx_sm[i], dy_sm_vals[i])

    # ── Foot positions ────────────────────────────────────────────────────
    lf = {'x': round(float(row['lax']), 1), 'y': round(float(row['lay']), 1)} \
         if not math.isnan(row['lax']) else None
    rf = {'x': round(float(row['rax']), 1), 'y': round(float(row['ray']), 1)} \
         if not math.isnan(row['rax']) else None

    jh   = round(float(row['jump_height_px']), 2)
    step = classify_step(jh, spd, acc)

    # ── Status flags ──────────────────────────────────────────────────────
    is_moving    = spd > 0.3
    is_jumping   = jh  > 15
    is_recovering = bool(row.get('is_burst', False))  # frame is post-burst

    frame_entries.append({
        'frame_id':  int(row['frame_id']),
        'timestamp': round(float(row['timestamp']), 4),
        'bounding_box': bb,
        'pose': {
            'keypoints': keypoints
        },
        'center_position': {
            'court_x': court_x,
            'court_y': court_y,
        },
        'movement': {
            'speed':        spd,
            'acceleration': acc,
            'direction':    dirn,
        },
        'footwork': {
            'step_type':  step,
            'left_foot':  lf,
            'right_foot': rf,
        },
        'status': {
            'is_moving':    is_moving,
            'is_jumping':   is_jumping,
            'is_recovering': is_recovering,
        },
    })

print(f'✅ Built {len(frame_entries)} frame entries')


In [ ]:
# ── Total distance covered (metres) ──────────────────────────────────────────
dx_m_all = df['foot_x_sm'].diff().fillna(0) / PX_PER_M_X
dy_m_all = pd.Series(savgol_filter(df['foot_y'].ffill().bfill(), WIN, 2)).diff().fillna(0) / PX_PER_M_Y
total_distance = float(np.sum(np.sqrt(dx_m_all**2 + dy_m_all**2)))

# ── Speed stats ───────────────────────────────────────────────────────────────
avg_speed = float(df['speed_mps'].mean())
max_speed = float(df['speed_mps'].max())

# ── Movement Efficiency (overall clip) ───────────────────────────────────────
net_displacement = float(np.sqrt(
    ((df['foot_x_sm'].iloc[-1] - df['foot_x_sm'].iloc[0]) / PX_PER_M_X)**2 +
    ((df['foot_y'].iloc[-1]    - df['foot_y'].iloc[0])    / PX_PER_M_Y)**2
))
overall_mei = round(min(net_displacement / total_distance, 1.0), 3) if total_distance > 0 else 1.0

# ── Court coverage % ─────────────────────────────────────────────────────────
# Divide court into 10×10 grid; count unique cells visited
GRID = 10
cx_norm = (df['foot_x'].dropna() / FRAME_W * GRID).astype(int).clip(0, GRID-1)
cy_norm = (df['foot_y'].dropna() / FRAME_H * GRID).astype(int).clip(0, GRID-1)
cells_visited = len(set(zip(cx_norm, cy_norm)))
court_coverage_pct = round(cells_visited / GRID**2 * 100, 1)

# ── Jump count ────────────────────────────────────────────────────────────────
jump_count = int(len(jump_peaks))

# ── Average recovery time (seconds between burst end and speed < threshold) ──
BURST_END_SPEED = df['speed_mps'].quantile(0.40)
recovery_times  = []
speed_vals      = df['speed_mps'].fillna(0).values
timestamps_arr  = df['timestamp'].values

for bf in burst_frames:
    for j in range(bf, min(bf + int(FPS * 3), len(speed_vals))):
        if speed_vals[j] < BURST_END_SPEED:
            recovery_times.append(timestamps_arr[j] - timestamps_arr[bf])
            break

avg_recovery_time = round(float(np.mean(recovery_times)), 3) if recovery_times else 0.0

# ── Pose stability score ──────────────────────────────────────────────────────
# Defined as 1 - normalised std of hip midpoint Y (lower oscillation = more stable)
hip_y_std    = df['hip_y'].std()
hip_y_range  = df['hip_y'].max() - df['hip_y'].min()
pose_stability = round(float(1.0 - min(hip_y_std / (hip_y_range + 1e-6), 1.0)), 3)

aggregated_metrics = {
    'total_distance_covered':   round(total_distance, 2),
    'average_speed':            round(avg_speed, 3),
    'max_speed':                round(max_speed, 3),
    'movement_efficiency':      overall_mei,
    'court_coverage_percentage': court_coverage_pct,
    'jump_count':               jump_count,
    'average_recovery_time':    avg_recovery_time,
    'pose_stability_score':     pose_stability,
}

print('Aggregated Metrics:')
for k, v in aggregated_metrics.items():
    print(f'  {k}: {v}')


In [ ]:
output_doc = {
    'match_id':  MATCH_ID,
    'player_id': PLAYER_ID,
    'frames':    frame_entries,
    'aggregated_metrics': aggregated_metrics,
}

with open(OUTPUT_JSON_PATH, 'w') as f:
    json.dump(output_doc, f, indent=2)

print(f'✅ JSON saved → {OUTPUT_JSON_PATH}')
print(f'   Frames: {len(frame_entries)}')
print(f'   Size  : {os.path.getsize(OUTPUT_JSON_PATH) / 1024:.1f} KB')


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
# Point this at your ORIGINAL video (same one fed into the pose pipeline)
# VIDEO_PATH is already defined if you ran the pose notebook in the same session.
# If not, set it manually:

VIDEO_PATH = "Mindblowing.mp4"

ANNOTATED_VIDEO_PATH = 'player_movement_annotated.mp4'

# Build a frame_id → entry lookup for O(1) access
frame_lookup = {e['frame_id']: e for e in frame_entries}

# ── Colour palette ────────────────────────────────────────────────────────────
COL = {
    'tracked':    (0, 220, 0),
    'predicted':  (0, 165, 255),
    'jump':       (0, 0, 255),
    'sprint':     (255, 100, 0),
    'lunge':      (180, 0, 255),
    'step':       (0, 200, 200),
    'stand':      (180, 180, 180),
    'skeleton':   (0, 255, 255),
    'kp_dot':     (0, 0, 255),
    'text_bg':    (20, 20, 20),
    'text_white': (255, 255, 255),
    'speed_bar':  (0, 200, 100),
    'accel_pos':  (0, 180, 255),
    'accel_neg':  (0, 60, 255),
}

SKELETON = [
    ('left_shoulder','left_elbow'),   ('left_elbow','left_wrist'),
    ('right_shoulder','right_elbow'), ('right_elbow','right_wrist'),
    ('left_shoulder','right_shoulder'),
    ('left_shoulder','left_hip'),     ('right_shoulder','right_hip'),
    ('left_hip','right_hip'),
    ('left_hip','left_knee'),         ('left_knee','left_ankle'),
    ('right_hip','right_knee'),       ('right_knee','right_ankle'),
    ('nose','left_eye'),              ('nose','right_eye'),
]

def draw_skeleton(frame, keypoints):
    kp_map = {kp['name']: (int(kp['x']), int(kp['y'])) for kp in keypoints}
    for a, b in SKELETON:
        if a in kp_map and b in kp_map:
            cv2.line(frame, kp_map[a], kp_map[b], COL['skeleton'], 2)
    for pt in kp_map.values():
        cv2.circle(frame, pt, 4, COL['kp_dot'], -1)

def draw_text_box(frame, lines, origin, font_scale=0.5, thickness=1, padding=5):
    """Draw a dark semi-transparent box then white text."""
    font   = cv2.FONT_HERSHEY_SIMPLEX
    line_h = int(font_scale * 28)
    max_w  = max(cv2.getTextSize(l, font, font_scale, thickness)[0][0] for l in lines)
    x0, y0 = origin
    box_h  = line_h * len(lines) + padding * 2
    box_w  = max_w + padding * 2

    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + box_w, y0 + box_h), COL['text_bg'], -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    for i, line in enumerate(lines):
        y = y0 + padding + (i + 1) * line_h - 4
        cv2.putText(frame, line, (x0 + padding, y), font, font_scale,
                    COL['text_white'], thickness, cv2.LINE_AA)

def draw_speed_bar(frame, speed, max_spd=10.0, origin=(20, 80), bar_w=180, bar_h=14):
    x, y = origin
    fill  = int(min(speed / max_spd, 1.0) * bar_w)
    cv2.rectangle(frame, (x, y), (x + bar_w, y + bar_h), (60, 60, 60), -1)
    cv2.rectangle(frame, (x, y), (x + fill,  y + bar_h), COL['speed_bar'], -1)
    cv2.rectangle(frame, (x, y), (x + bar_w, y + bar_h), (200, 200, 200), 1)
    cv2.putText(frame, f'{speed:.1f} m/s', (x + bar_w + 6, y + bar_h - 2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, COL['text_white'], 1, cv2.LINE_AA)

def draw_direction_arrow(frame, cx, cy, direction_deg, speed, length=40):
    if speed < 0.3:
        return
    rad  = math.radians(direction_deg)
    ex   = int(cx + length * math.cos(rad))
    ey   = int(cy - length * math.sin(rad))   # image Y inverted
    cv2.arrowedLine(frame, (cx, cy), (ex, ey), (255, 220, 0), 2, tipLength=0.3)

# ── Process video ─────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), 'Cannot open video — check VIDEO_PATH'

fw = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
fh = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_out = cap.get(cv2.CAP_PROP_FPS) or 30
total_fr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out_vid = cv2.VideoWriter(
    ANNOTATED_VIDEO_PATH,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps_out, (fw, fh)
)

print(f'Writing annotated video ({total_fr} frames)...')

for frame_id in tqdm(range(total_fr)):
    ret, frame = cap.read()
    if not ret:
        break

    entry = frame_lookup.get(frame_id)

    if entry is None:
        # No detection this frame — just write plain
        out_vid.write(frame)
        continue

    bb    = entry['bounding_box']
    mv    = entry['movement']
    fw_   = entry['footwork']
    st    = entry['status']
    cp    = entry['center_position']
    kps   = entry['pose']['keypoints']

    x1 = int(bb['x'])
    y1 = int(bb['y'])
    x2 = int(bb['x'] + bb['width'])
    y2 = int(bb['y'] + bb['height'])
    cx = (x1 + x2) // 2
    cy = (y1 + y2) // 2

    step_col = COL.get(fw_['step_type'], COL['stand'])

    # 1. Bounding box
    cv2.rectangle(frame, (x1, y1), (x2, y2), step_col, 2)

    # 2. Skeleton
    if kps:
        draw_skeleton(frame, kps)

    # 3. Direction arrow from hip centre
    draw_direction_arrow(frame, cx, cy, mv['direction'], mv['speed'])

    # 4. Per-player info box (top-left of bbox)
    zone = entry.get('court_zone', '')  # may not be in entry; pulled below
    zone_label = f"Zone: {df.loc[df['frame_id']==frame_id, 'court_zone'].values[0] if frame_id in df['frame_id'].values else '?'}"

    flags = []
    if st['is_jumping']:   flags.append('JUMP')
    if st['is_recovering']:flags.append('RECOVER')
    if not flags and st['is_moving']: flags.append(fw_['step_type'].upper())
    if not flags:          flags.append('STAND')

    lines = [
        f"Spd: {mv['speed']:.1f} m/s  Acc: {mv['acceleration']:+.1f}",
        f"Dir: {mv['direction']:.0f}deg  Step: {fw_['step_type']}",
        f"Court: ({cp['court_x']}, {cp['court_y']}) m",
        zone_label,
        '  '.join(flags),
    ]
    draw_text_box(frame, lines, (max(0, x1), max(0, y1 - 110)))

    # 5. Speed bar (top-left corner of frame)
    draw_speed_bar(frame, mv['speed'], origin=(16, 20))

    # 6. Foot dots
    for foot_key in ('left_foot', 'right_foot'):
        ft = fw_.get(foot_key)
        if ft:
            cv2.circle(frame, (int(ft['x']), int(ft['y'])), 6, (0, 255, 180), -1)

    # 7. Jump height indicator (right side of bbox)
    jh_row = df.loc[df['frame_id'] == frame_id, 'jump_height_px']
    if not jh_row.empty and jh_row.values[0] > 10:
        jh_px  = int(jh_row.values[0])
        jh_bar_x = x2 + 6
        cv2.line(frame, (jh_bar_x, y2), (jh_bar_x, max(y2 - jh_px, 0)),
                 (0, 0, 255), 4)
        cv2.putText(frame, f'J:{jh_px}px', (jh_bar_x + 4, max(y2 - jh_px - 4, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.38, (0, 0, 255), 1, cv2.LINE_AA)

    out_vid.write(frame)

cap.release()
out_vid.release()
print(f'✅ Annotated video saved → {ANNOTATED_VIDEO_PATH}')


In [ ]:
print('📥 Downloading structured JSON...')
print('📥 Downloading annotated video...')
print('\n✅ Both files downloaded!')
print(f'   JSON : {os.path.getsize(OUTPUT_JSON_PATH)  / 1024:.1f} KB')
print(f'   Video: {os.path.getsize(ANNOTATED_VIDEO_PATH) / (1024*1024):.1f} MB')
